In [1]:
#1: Setup and first look
import pandas as pd
df=pd.read_csv('data/netflix_titles.csv')

In [2]:
print("Shape of Dataset:")
print(df.shape)
print("Head:")
print(df.head())
print("Dataset info")
print(df.info())
print("Null values")
print(df.isnull().sum())

Shape of Dataset:
(8807, 12)
Head:
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seaso

In [3]:
#2: Data cleaning

#filling missing director, cast and country with 'Unknown' instead of deleting values
df['director']=df['director'].fillna("Unknown")
df['cast']=df['cast'].fillna("Unknown")
df['country']=df['country'].fillna("Unknown")

#dropping rows with missing date_added, rating, duration as veery little are missing
df=df.dropna(subset=['date_added','duration','rating'])
df.isnull().sum()

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64

In [4]:
#converting date to datetime from string and extracting month and year
#df['date_added'] = pd.to_datetime(df['date_added'].str.strip() if df['date_added'].dtype=='object' else df['date_added'])
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())
df['year_added']=df['date_added'].dt.year
df['month_added']=df['date_added'].dt.month

#checking for duplicates
df.duplicated().sum()
df=df.drop_duplicates()

df['type'].value_counts()
df['rating'].unique()

<ArrowStringArray>
[   'PG-13',    'TV-MA',       'PG',    'TV-14',    'TV-PG',     'TV-Y',
    'TV-Y7',        'R',     'TV-G',        'G',    'NC-17',       'NR',
 'TV-Y7-FV',       'UR']
Length: 14, dtype: str

In [5]:
#3: Feature engineering

#splitting duration into numeric minutes(movies) and seasons(series)
df['duration_int']=df['duration'].str.extract('(\d+)').astype(float)

df['duration_minutes']=df.apply( lambda row: row['duration_int'] if row['type']=='Movie' else None, axis=1)
df['duration_seasons']=df.apply( lambda row: row['duration_int'] if row['type']=='TV Show' else None, axis=1)

#extracting first listed country and genre
df['primary_country']=df['country'].apply( lambda x: x.split(',')[0].strip())
df['primary_genre']=df['listed_in'].apply( lambda x: x.split(',')[0].strip())

df[['type','duration','duration_minutes','duration_seasons']].sample(10)
print(df['primary_country'].value_counts().head(10) )
print()
df['primary_genre'].value_counts().head(10)

primary_country
United States     3202
India             1008
Unknown            829
United Kingdom     627
Canada             271
Japan              257
France             212
South Korea        211
Spain              181
Mexico             134
Name: count, dtype: int64



primary_genre
Dramas                      1599
Comedies                    1210
Action & Adventure           859
Documentaries                829
International TV Shows       773
Children & Family Movies     605
Crime TV Shows               399
Kids' TV                     385
Stand-Up Comedy              334
Horror Movies                275
Name: count, dtype: int64

In [29]:
#4: Exploratory Analysis
content_by_year=df['year_added'].value_counts().sort_index()
#print(content_by_year)
type_by_year=df.groupby(['year_added','type']).size().unstack()
#print(type_by_year)
top_countries = df[df['primary_country'] != 'Unknown']['primary_country'].value_counts().head(10)
#print(top_countries)
genre_trend=df.groupby(['year_added','primary_genre']).size().unstack(fill_value=0)
top_5_genres=df['primary_genre'].value_counts().head(5).index
#genre_trend[top_5_genres]
#df['rating'].value_counts()
#df['duration_minutes'].describe()
#df['duration_seasons'].describe()

In [30]:
#5: Exporting and connecting to Tablaeu
df.to_csv('data/netflix_cleaned.csv', index=False)